# Comparaison simple : facteurs historiques et composites améliorés

Ce notebook compare, séparément pour **STOXX EUROPE 600** et **EUROPE SMALL CAP** :

1. le facteur historique du screen ;
2. un composite amélioré qui combine ce facteur historique avec une ou plusieurs nouvelles variables ;
3. l'effet incrémental de chaque nouvelle variable ajoutée seule au facteur historique ;
4. les performances Top, les ratios relatifs et les figures Plotly.

Les cellules `COMPOSITE_CONFIGS_STOXX` et `COMPOSITE_CONFIGS_SMALL` sont les seules cellules de paramétrage à modifier en priorité. Les poids sont relatifs : ils peuvent être changés manuellement sans modifier le pipeline. Les commentaires, messages et cette documentation restent volontairement en français pour conserver la convention du projet.

Les résultats sont écrits dans `exports/factor_core_upgrade_comparison_STOXX600` et `exports/factor_core_upgrade_comparison_SMALL`. Le notebook est livré sans sorties d'exécution.

In [ ]:
from pathlib import Path
import json
import re

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

from factor_config import LOWER_IS_BETTER, signal_options
from func import (
    calculate_benchmark_performance,
    calculate_performance_ratios,
    combine_backtest_performances,
    export_backtest_results,
    load_backtest_data,
    plot_performance_comparison,
    test_composite_signals,
    test_incremental_signals,
)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'func.py').exists():
    raise RuntimeError(
        'Définissez le répertoire de travail Jupyter sur C:\\dev\\factor_backtest.'
    )

DATA_DIR = REPO_ROOT / 'data'
SCREEN_PATH = DATA_DIR / 'screen_aggregate.parquet'
RETURNS_PATH = DATA_DIR / 'returns.parquet'
EXPORT_ROOT = REPO_ROOT / 'exports'
LIST_NOIRE_PATH = None
START_DATE = '2007-12-01'
PERCENTILE = 0.13
N_JOBS = 1
PERIOD_BREAKPOINTS = [2009, 2013, 2017, 2020, 2022, 2024, 2026]

METRIC_COLUMNS = [
    'active_cagr',
    'top_worst_cagr',
    'top_information_ratio',
    'robust_score',
    'active_max_drawdown',
    'tracking_error_annualized',
    'min_rolling_3y_cagr',
    'top_bench_ratio',
    'top_worst_ratio',
    'top_annualized_return',
    'observation_count',
    'years',
]


def safe_name(value):
    """Produit un identifiant court pour les fichiers et les noms de groupes."""
    return re.sub(r'[^A-Za-z0-9._-]+', '_', str(value)).strip('._') or 'variable'


def resolve_baseline_columns(candidates, available_columns):
    """Choisit le premier ancien facteur réellement présent dans le screen."""
    resolved = {}
    for family, options in candidates.items():
        selected = next((name for name in options if name in available_columns), None)
        if selected is None:
            raise KeyError(
                f'Aucun facteur historique disponible pour {family}: {options}'
            )
        resolved[family] = selected
    return resolved


def make_signal_config(specs):
    """Transforme une liste lisible de composantes en configuration du moteur."""
    config = {}
    for spec in specs:
        variable = spec['variable']
        dimension = spec['dimension']
        higher_is_better = bool(spec.get('higher_is_better', variable not in LOWER_IS_BETTER))
        expected_direction = variable not in LOWER_IS_BETTER
        if higher_is_better != expected_direction:
            raise ValueError(
                f'Direction incohérente pour {variable}: utilisez higher_is_better={expected_direction}.'
            )
        options = config.setdefault(
            variable,
            signal_options(higher_is_better=higher_is_better),
        )
        options[f'weight_{dimension}'] = float(spec.get('weight', 1.0))
    return config


def make_baseline_config(variable):
    """Construit le facteur historique seul, au niveau du screen."""
    return {variable: signal_options(level=1.0, higher_is_better=True)}


def collect_variables(baseline_columns, composite_specs):
    """Retourne les colonnes nécessaires sans doublons."""
    variables = list(baseline_columns.values())
    variables.extend(
        spec['variable']
        for specs in composite_specs.values()
        for spec in specs
    )
    return list(dict.fromkeys(variables))


def comparison_table(metrics, old_path, upgrade_path, family, upgrade_label='upgrade'):
    """Compare les métriques historiques et améliorées période par période."""
    keys = [column for column in ('scope', 'period_id', 'period_label') if column in metrics.columns]
    columns = [column for column in METRIC_COLUMNS if column in metrics.columns]
    old = metrics.loc[metrics['test_path'].eq(old_path), keys + columns].copy()
    upgrade = metrics.loc[metrics['test_path'].eq(upgrade_path), keys + columns].copy()
    if old.empty or upgrade.empty:
        raise KeyError(f'Métriques introuvables pour {family}: {old_path} / {upgrade_path}')
    old = old.rename(columns={column: f'{column}_old' for column in columns})
    upgrade = upgrade.rename(columns={column: f'{column}_{upgrade_label}' for column in columns})
    joined = upgrade.merge(old, on=keys, how='outer')
    joined.insert(0, 'family', family)
    for column in columns:
        joined[f'delta_{column}'] = joined[f'{column}_{upgrade_label}'] - joined[f'{column}_old']
    required = {'active_cagr', 'top_worst_cagr', 'top_information_ratio'}
    if required.issubset(columns):
        joined['upgrade_gate'] = (
            joined[f'active_cagr_{upgrade_label}'].gt(0)
            & joined[f'top_worst_cagr_{upgrade_label}'].gt(0)
            & joined[f'top_information_ratio_{upgrade_label}'].gt(0)
        )
        joined['old_gate'] = (
            joined['active_cagr_old'].gt(0)
            & joined['top_worst_cagr_old'].gt(0)
            & joined['top_information_ratio_old'].gt(0)
        )
        joined['performance_improved'] = (
            joined[f'delta_active_cagr'].gt(0)
            & joined[f'delta_top_worst_cagr'].gt(0)
            & joined[f'delta_top_information_ratio'].gt(0)
        )
    if {'active_max_drawdown', 'tracking_error_annualized'}.issubset(columns):
        joined['risk_not_worse'] = (
            joined['delta_active_max_drawdown'].le(0)
            & joined['delta_tracking_error_annualized'].le(0)
        )
    sort_columns = [column for column in ('period_id', 'scope') if column in joined.columns]
    return joined.sort_values(sort_columns).reset_index(drop=True)


def run_market(market, benchmark, output_name, baseline_candidates, composite_specs):
    """Exécute un marché : composites, incrémentaux, métriques et manifest."""
    available_columns = set(pq.ParquetFile(SCREEN_PATH).schema_arrow.names)
    baseline_columns = resolve_baseline_columns(baseline_candidates, available_columns)
    load_variables = collect_variables(baseline_columns, composite_specs)
    missing = [column for column in load_variables if column not in available_columns]
    if missing:
        raise KeyError(f'Variables absentes du screen pour {market}: {missing}')

    screen, returns = load_backtest_data(
        screen_path=SCREEN_PATH,
        returns_path=RETURNS_PATH,
        variables=load_variables,
        bench=benchmark,
        start_date=START_DATE,
        lookback_periods=12,
        compact_dtypes=True,
    )
    screen['Date'] = pd.to_datetime(screen['Date'])
    if f'Weight in {benchmark}' not in screen.columns:
        raise KeyError(f'La colonne Weight in {benchmark} est absente du screen.')
    benchmark_performance = calculate_benchmark_performance(
        screen=screen, returns=returns, bench=benchmark, start_date=START_DATE
    )
    run_options = {
        'bench': benchmark,
        'bench_perf': benchmark_performance,
        'percentile': PERCENTILE,
        'start_date': START_DATE,
        'freq_rebal': 1,
        'fill_method': 'copy',
        'n_jobs': N_JOBS,
        'retain_builders': False,
        'monthly_base_cache': {},
        'period_breakpoints': PERIOD_BREAKPOINTS,
        'show_plot': False,
        'build_figure': False,
    }

    composite_configs = {}
    for family, specs in composite_specs.items():
        for spec in specs:
            if spec['variable'] not in screen.columns:
                raise KeyError(f'{spec["variable"]} absent après chargement pour {market}.')
        composite_configs[f'old__{family}'] = make_baseline_config(baseline_columns[family])
        composite_configs[f'upgrade__{family}'] = make_signal_config(specs)

    composite_batch = test_composite_signals(
        screen=screen,
        returns=returns,
        composite_configs=composite_configs,
        list_noire_path=LIST_NOIRE_PATH,
        score_prefix=f'Score_CoreUpgrade_{safe_name(market)}',
        **run_options,
    )

    incremental_batches = {}
    incremental_specs = {}
    working_screen = composite_batch['screen']
    for family, specs in composite_specs.items():
        satellites = [spec for spec in specs if spec['variable'] != baseline_columns[family]]
        for index, spec in enumerate(satellites, start=1):
            group_key = f'{family}__{index}__{safe_name(spec["variable"])}__{spec["dimension"]}'
            incremental_specs[group_key] = {'family': family, **spec}
            incremental_batches[group_key] = test_incremental_signals(
                screen=working_screen,
                returns=returns,
                baseline_config=make_baseline_config(baseline_columns[family]),
                candidate_config=make_signal_config([spec]),
                list_noire_path=LIST_NOIRE_PATH,
                **run_options,
            )
            working_screen = incremental_batches[group_key]['screen']

    all_results = {'composites': composite_batch, 'incremental': incremental_batches}
    exported = export_backtest_results(
        results=all_results,
        output_dir=EXPORT_ROOT,
        export_name=output_name,
        export_html=False,
        export_png=False,
        export_holdings=False,
    )
    export_dir = Path(exported['export_dir'])
    metrics = exported['metrics'].copy()
    registry = exported['registry']
    path_by_name = {
        entry.get('metadata', {}).get('test_name'): entry.get('test_path')
        for entry in registry
        if entry.get('metadata', {}).get('test_name') and entry.get('test_path')
    }

    family_tables = []
    for family in composite_specs:
        table = comparison_table(
            metrics,
            path_by_name[f'old__{family}'],
            path_by_name[f'upgrade__{family}'],
            family,
        )
        family_tables.append(table)
    family_comparison = pd.concat(family_tables, ignore_index=True)
    family_comparison.to_csv(export_dir / 'family_composite_vs_old.csv', index=False)
    family_comparison.loc[family_comparison['period_id'].eq('total')].to_csv(
        export_dir / 'family_composite_vs_old_total.csv', index=False
    )

    incremental_tables = []
    for group_key, spec in incremental_specs.items():
        prefix = f'incremental / {group_key} / '
        entries = [entry for entry in registry if entry.get('test_path', '').startswith(prefix)]
        baseline_entry = next(
            (entry for entry in entries if entry.get('metadata', {}).get('test_name') == 'Baseline'),
            None,
        )
        candidate_entry = next(
            (entry for entry in entries if entry.get('metadata', {}).get('test_name') == spec['variable']),
            None,
        )
        if baseline_entry is None or candidate_entry is None:
            raise KeyError(f'Résultat incrémental incomplet pour {group_key}.')
        table = comparison_table(
            metrics,
            baseline_entry['test_path'],
            candidate_entry['test_path'],
            spec['family'],
            upgrade_label='candidate',
        )
        table.insert(1, 'variable', spec['variable'])
        table.insert(2, 'dimension', spec['dimension'])
        table.insert(3, 'weight', spec.get('weight', 1.0))
        table.insert(4, 'incremental_group', group_key)
        incremental_tables.append(table)
    incremental_effects = (
        pd.concat(incremental_tables, ignore_index=True) if incremental_tables else pd.DataFrame()
    )
    incremental_effects.to_csv(export_dir / 'incremental_effects.csv', index=False)
    if not incremental_effects.empty:
        incremental_effects.loc[incremental_effects['period_id'].eq('total')].to_csv(
            export_dir / 'incremental_effects_total.csv', index=False
        )

    manifest_rows = []
    for family, specs in composite_specs.items():
        manifest_rows.append({
            'market': market, 'family': family, 'role': 'old_factor',
            'variable': baseline_columns[family], 'dimension': 'level', 'weight': 1.0,
        })
        for index, spec in enumerate(specs, start=1):
            manifest_rows.append({
                'market': market, 'family': family, 'role': spec.get('role', f'component_{index}'),
                'variable': spec['variable'], 'dimension': spec['dimension'],
                'weight': spec.get('weight', 1.0),
                'higher_is_better': spec.get('higher_is_better'),
            })
    pd.DataFrame(manifest_rows).to_csv(export_dir / 'composite_config_manifest.csv', index=False)

    return {
        'market': market,
        'benchmark': benchmark,
        'baseline_columns': baseline_columns,
        'composite_specs': composite_specs,
        'incremental_specs': incremental_specs,
        'results': all_results,
        'export_dir': export_dir,
        'metrics': metrics,
        'registry': registry,
        'path_by_name': path_by_name,
        'family_comparison': family_comparison,
        'incremental_effects': incremental_effects,
        'period_breakpoints': PERIOD_BREAKPOINTS,
    }


def plot_market_comparisons(bundle):
    """Crée une figure interactive par famille, ancien facteur contre upgrade."""
    figures_dir = bundle['export_dir'] / 'figures'
    data_dir = bundle['export_dir'] / 'data'
    figures_dir.mkdir(parents=True, exist_ok=True)
    data_dir.mkdir(parents=True, exist_ok=True)
    figures = {}
    for family in bundle['composite_specs']:
        old_path = bundle['path_by_name'][f'old__{family}']
        upgrade_path = bundle['path_by_name'][f'upgrade__{family}']
        selections = {
            f'Ancien | {family}': (old_path, 'Top'),
            f'Upgrade | {family}': (upgrade_path, 'Top'),
            'Benchmark': (old_path, 'Bench'),
        }
        performance, composition = combine_backtest_performances(
            export_dir=bundle['export_dir'],
            selections=selections,
            return_composition=True,
        )
        ratios = calculate_performance_ratios(performance, benchmark_column='Benchmark')
        performance.to_csv(data_dir / f'{family}_old_vs_upgrade_performance.csv')
        composition.to_csv(data_dir / f'{family}_old_vs_upgrade_composition.csv', index=False)
        figure = plot_performance_comparison(
            performance=performance,
            ratios=ratios,
            benchmark_column='Benchmark',
            title=f"{bundle['market']} | {family} : ancien contre upgrade",
            save_path=figures_dir / f'{family}_old_vs_upgrade.html',
            show_plot=False,
            rebase=True,
            period_breakpoints=bundle['period_breakpoints'],
            default_period_id='total',
        )
        figures[family] = figure
        display(figure)
    print(f"Figures enregistrées dans : {figures_dir}")
    return figures


## 1. STOXX EUROPE 600

Le facteur historique est recherché en priorité sous la forme `Avg Percentile`, qui correspond aux résultats historiques collés dans la recherche. Les composantes nouvelles ci-dessous sont des points de départ issus du rapport d'évidence récent ; elles ne sont pas figées. Modifiez uniquement la liste de composantes et leurs poids avant d'exécuter la cellule suivante.

Chaque ligne de `COMPOSITE_CONFIGS_STOXX` contient `variable`, `dimension`, `weight` et `higher_is_better`. Le poids de l'ancien facteur est également visible dans la configuration et peut être modifié.

In [ ]:
# CELLULE À AJUSTER MANUELLEMENT : configuration STOXX.
STOXX_BASELINE_CANDIDATES = {
    'growth': ('Growth Avg Percentile', 'GROWTH_SCORE_FS_SECTOR'),
    'quality': ('Quality Avg Percentile', 'MARGIN_SCORE_FS_SECTOR'),
    'momentum': ('Mom Avg Percentile', 'MOMENTUM_SCORE_FS_SECTOR'),
    'value': ('Value Avg Percentile', 'VALUE_SCORE_FS_SECTOR'),
    'dividend': ('Dividend Avg Percentile', 'Dividend_NTM Avg Percentile'),
}
STOXX_AVAILABLE = set(pq.ParquetFile(SCREEN_PATH).schema_arrow.names)
STOXX_BASELINE_COLUMNS = resolve_baseline_columns(STOXX_BASELINE_CANDIDATES, STOXX_AVAILABLE)
STOXX_OLD = STOXX_BASELINE_COLUMNS

COMPOSITE_CONFIGS_STOXX = {
    'quality': [
        {'role': 'old_core', 'variable': STOXX_OLD['quality'], 'dimension': 'level', 'weight': 0.75, 'higher_is_better': True},
        {'role': 'satellite', 'variable': 'NetDebt to EBITDA exFIN', 'dimension': 'rank_diff_3', 'weight': 0.25, 'higher_is_better': False},
    ],
    'growth': [
        {'role': 'old_core', 'variable': STOXX_OLD['growth'], 'dimension': 'level', 'weight': 0.75, 'higher_is_better': True},
        {'role': 'satellite', 'variable': 'CFO 5Y CAGR', 'dimension': 'level', 'weight': 0.25, 'higher_is_better': True},
    ],
    'momentum': [
        {'role': 'old_core', 'variable': STOXX_OLD['momentum'], 'dimension': 'level', 'weight': 0.75, 'higher_is_better': True},
        {'role': 'satellite', 'variable': 'SP Price Target CIQ', 'dimension': 'pct_12', 'weight': 0.25, 'higher_is_better': True},
    ],
    'dividend': [
        {'role': 'old_core', 'variable': STOXX_OLD['dividend'], 'dimension': 'level', 'weight': 0.60, 'higher_is_better': True},
        {'role': 'satellite', 'variable': 'CFO Div Cov Ratio', 'dimension': 'diff_3', 'weight': 0.20, 'higher_is_better': True},
        {'role': 'satellite', 'variable': 'DPS FY1', 'dimension': 'pct_6', 'weight': 0.20, 'higher_is_better': True},
    ],
    'value': [
        {'role': 'old_core', 'variable': STOXX_OLD['value'], 'dimension': 'level', 'weight': 0.70, 'higher_is_better': True},
        {'role': 'satellite', 'variable': 'Earns Yield FY1', 'dimension': 'diff_6', 'weight': 0.15, 'higher_is_better': True},
        {'role': 'satellite', 'variable': 'EV To EBITDA LTM', 'dimension': 'pct_3', 'weight': 0.15, 'higher_is_better': False},
    ],
}

display(pd.DataFrame(
    [
        {'family': family, **spec}
        for family, specs in COMPOSITE_CONFIGS_STOXX.items()
        for spec in specs
    ]
).loc[:, ['family', 'role', 'variable', 'dimension', 'weight', 'higher_is_better']])

STOXX_BUNDLE = run_market(
    market='STOXX EUROPE 600',
    benchmark='STOXX EUROPE 600',
    output_name='factor_core_upgrade_comparison_STOXX600',
    baseline_candidates=STOXX_BASELINE_CANDIDATES,
    composite_specs=COMPOSITE_CONFIGS_STOXX,
)

display(STOXX_BUNDLE['family_comparison'].loc[STOXX_BUNDLE['family_comparison']['period_id'].eq('total')])
display(STOXX_BUNDLE['incremental_effects'].loc[STOXX_BUNDLE['incremental_effects']['period_id'].eq('total')])
plot_market_comparisons(STOXX_BUNDLE)


### Lecture des sorties STOXX

- `family_composite_vs_old.csv` contient la comparaison par période entre l'ancien facteur et l'upgrade. Les colonnes `delta_*` sont calculées comme `upgrade - old`.
- `incremental_effects.csv` mesure l'ajout d'une seule nouvelle variable au facteur historique. Une variable ne doit pas être conservée sur le seul CAGR : vérifier aussi `top_worst_cagr`, `top_information_ratio`, le drawdown actif et le tracking error.
- `figures/<family>_old_vs_upgrade.html` contient la courbe Top de l'ancien facteur, la courbe Top de l'upgrade et le benchmark, avec le sélecteur de période.

## 2. EUROPE SMALL CAP

Cette section est volontairement indépendante de STOXX : elle recharge le même screen, utilise le benchmark `MSCI EUR SMALL`, mais possède ses propres facteurs historiques et sa propre configuration composite. Les poids et les variables peuvent être changés sans toucher à la section STOXX.

In [ ]:
# CELLULE À AJUSTER MANUELLEMENT : configuration EUROPE SMALL CAP.
SMALL_BASELINE_CANDIDATES = {
    'growth': ('Growth Avg Percentile', 'GROWTH_SCORE_FS_SECTOR'),
    'quality': ('Quality Avg Percentile', 'MARGIN_SCORE_FS_SECTOR'),
    'momentum': ('Mom Avg Percentile', 'MOMENTUM_SCORE_FS_SECTOR'),
    'value': ('Value Avg Percentile', 'VALUE_SCORE_FS_SECTOR'),
    'dividend': ('Dividend Avg Percentile', 'Dividend_NTM Avg Percentile'),
}
SMALL_AVAILABLE = set(pq.ParquetFile(SCREEN_PATH).schema_arrow.names)
SMALL_BASELINE_COLUMNS = resolve_baseline_columns(SMALL_BASELINE_CANDIDATES, SMALL_AVAILABLE)
SMALL_OLD = SMALL_BASELINE_COLUMNS

COMPOSITE_CONFIGS_SMALL = {
    'quality': [
        {'role': 'old_core', 'variable': SMALL_OLD['quality'], 'dimension': 'level', 'weight': 0.75, 'higher_is_better': True},
        {'role': 'satellite', 'variable': 'PCT ROE', 'dimension': 'diff_3', 'weight': 0.25, 'higher_is_better': True},
    ],
    'growth': [
        {'role': 'old_core', 'variable': SMALL_OLD['growth'], 'dimension': 'level', 'weight': 0.70, 'higher_is_better': True},
        {'role': 'satellite', 'variable': 'PCT Hist GrossInc', 'dimension': 'rank_diff_3', 'weight': 0.30, 'higher_is_better': True},
    ],
    'momentum': [
        {'role': 'old_core', 'variable': SMALL_OLD['momentum'], 'dimension': 'level', 'weight': 0.75, 'higher_is_better': True},
        {'role': 'satellite', 'variable': 'SP Price Target CIQ', 'dimension': 'pct_6', 'weight': 0.25, 'higher_is_better': True},
    ],
    'dividend': [
        {'role': 'old_core', 'variable': SMALL_OLD['dividend'], 'dimension': 'level', 'weight': 0.70, 'higher_is_better': True},
        {'role': 'satellite', 'variable': 'PCT DvdYield FY1', 'dimension': 'level', 'weight': 0.30, 'higher_is_better': True},
    ],
    'value': [
        {'role': 'old_core', 'variable': SMALL_OLD['value'], 'dimension': 'level', 'weight': 0.70, 'higher_is_better': True},
        {'role': 'satellite', 'variable': 'Earns Yield FY0', 'dimension': 'level', 'weight': 0.20, 'higher_is_better': True},
        {'role': 'satellite', 'variable': 'EV To EBITDA LTM', 'dimension': 'rank_diff_3', 'weight': 0.10, 'higher_is_better': False},
    ],
}

display(pd.DataFrame(
    [
        {'family': family, **spec}
        for family, specs in COMPOSITE_CONFIGS_SMALL.items()
        for spec in specs
    ]
).loc[:, ['family', 'role', 'variable', 'dimension', 'weight', 'higher_is_better']])

SMALL_BUNDLE = run_market(
    market='EUROPE SMALL CAP',
    benchmark='MSCI EUR SMALL',
    output_name='factor_core_upgrade_comparison_SMALL',
    baseline_candidates=SMALL_BASELINE_CANDIDATES,
    composite_specs=COMPOSITE_CONFIGS_SMALL,
)

display(SMALL_BUNDLE['family_comparison'].loc[SMALL_BUNDLE['family_comparison']['period_id'].eq('total')])
display(SMALL_BUNDLE['incremental_effects'].loc[SMALL_BUNDLE['incremental_effects']['period_id'].eq('total')])
plot_market_comparisons(SMALL_BUNDLE)


### Lecture finale

Commencer par les deux fichiers `family_composite_vs_old_total.csv`, puis vérifier la stabilité dans les lignes par sous-période. Une amélioration utile doit idéalement conserver un gate Top/Worst/IR positif, améliorer plusieurs métriques relatives et ne pas dégrader simultanément le drawdown actif et le tracking error.

Les lignes `before_2009` ou les périodes sans observations doivent rester traitées comme absentes, et non comme une performance nulle. Le notebook ne promeut aucun facteur en production : il produit uniquement les tables et figures nécessaires à la revue de recherche.